In [ ]:
import pandas as pd
from datetime import datetime
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import load_targets

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "42646413"
countries = [p.parts[-1] for p in job_path.iterdir()]
spaghs = {}
targets = {}
for c in countries:
    country_path = job_path / c
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    country_spaghs = [pd.read_hdf(country_path / a / "spaghetti.h5") for a in analyses]
    spaghs[c] = pd.concat(country_spaghs, keys=analyses, axis=1)
    targets[c] = load_targets(country_path / analyses[0])

In [ ]:
for c in countries:
    c_spaghs = spaghs[c]
    c_targs = targets[c]
    country_path = job_path / c
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    fig, axes = plt.subplots(len(c_targs), len(analyses), figsize=[12, 15], sharex=True, sharey="row")
    fig.suptitle(c, fontsize=18, y=1.0)
    for a, analysis in enumerate(analyses):
        a_spaghs = c_spaghs[analysis]
        for o, out in enumerate(c_targs):
            ax = axes[o, a]
            o_spagh = a_spaghs[out]
            for col in o_spagh.columns:
                ax.plot(o_spagh.index, o_spagh[col], color="black", linewidth=0.2)
            target = c_targs[out]
            ax.plot(target.index, target, linewidth=0.0, marker=".")
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
            if o == 0:
                ax.set_title(analysis, fontsize=10)
            if a == 0:
                ax.set_ylabel(out, fontsize=10)
    fig.tight_layout()
    fig.subplots_adjust(wspace=0.05)